In [ ]:
import os
import tempfile
from pyaedt import Icepak, Desktop

def run_analysis():
    # Launch AEDT Desktop and create a new Icepak design.
    desktop = Desktop("2024.1", non_graphical=True, new_desktop=True)
    ipk = Icepak(project="CoolingRegionProject", design="CoolingEnclosure",
                 solution_type="SteadyState", version="2024.1")
    ipk.modeler.model_units = "mm"

    # --- Material Assignment ---
    # Add a custom silicon material (chip) if not already in the database.
    ipk.materials.add_material(
        name="silicon_custom",
        properties={
            "Thermal Conductivity": "130",  # W/m-K
            "Mass Density": "2329",         # kg/m³
            "Specific Heat": "700"          # J/kg-K
        }
    )
    # For ambient air, we assume a material named "air" exists.
    # (If not, you can add one similarly.)
    
    # --- Geometry Creation ---
    # Create an ambient region (the enclosure) made of air.
    air_region = ipk.modeler.create_box(
        origin=[0, 0, 0],
        sizes=[200, 150, 100],
        name="AirRegion",
        material="air"
    )
    # Create a chip (heat source) inside the region.
    chip = ipk.modeler.create_box(
        origin=[50, 50, 20],
        sizes=[20, 20, 2],
        name="Chip",
        material="silicon_custom"
    )
    
    # --- Boundary Assignment for Forced Convection ---
    # Using the sample snippet, we assume that the air region can be retrieved by name
    # and that it has attributes top_face_x and bottom_face_x.
    try:
        region_obj = ipk.modeler["AirRegion"]
        # Assign pressure-free opening at the top face (Outlet)
        ipk.assign_pressure_free_opening(
            assignment=region_obj.top_face_x.id, boundary_name="Outlet"
        )
        # Assign velocity free opening at the bottom face (Inlet)
        ipk.assign_velocity_free_opening(
            assignment=region_obj.bottom_face_x.id,
            boundary_name="Inlet",
            velocity=["1m_per_sec", "0m_per_sec", "0m_per_sec"]
        )
        print("Assigned inlet/outlet openings using AirRegion face properties.")
    except Exception as e:
        print("Error assigning openings using region face properties:", e)
    
    # --- Thermal Load Assignment ---
    # Assign a thermal source (Total Power) to the chip.
    ipk.assign_source("Chip", "Total Power", 100)  # 100 W thermal load
    
    # --- Export Design Preview ---
    # Save the project so that a preview image can be generated.
    ipk.save_project()
    preview_path = os.path.join(tempfile.gettempdir(), "cooling_enclosure_preview.jpg")
    try:
        ipk.export_design_preview_to_jpg(preview_path)
        print("Design preview image exported to:", preview_path)
    except Exception as e:
        print("Error exporting design preview image:", e)
    
    # --- Mesh Setup ---
    
    ipk.mesh.set_global_mesh_settings(max_element_size=20, min_element_size=2)
    # Refine mesh on the chip
    ipk.mesh.assign_mesh_control("Chip", max_element_size=5)
    
    # --- Analysis Setup and Run ---
    if not ipk.setup_names:
        setup = ipk.create_setup()
        setup.props["Convergence Criteria"] = 1e-6
        setup.props["Max Number Of Iterations"] = 1000
    ipk.analyze_all()
    
    # --- Retrieve and Print Results ---
    results = {}
    if hasattr(ipk.post, "get_temperature_data"):
        chip_temp = ipk.post.get_temperature_data("Chip")
        region_temp = ipk.post.get_temperature_data("AirRegion")
        if chip_temp:
            results["Chip"] = {"max_temperature": max(chip_temp)}
        if region_temp:
            results["AirRegion"] = {
                "max_temperature": max(region_temp),
                "min_temperature": min(region_temp),
                "avg_temperature": sum(region_temp) / len(region_temp)
            }
    
    print("\nThermal Analysis Results:")
    for comp, data in results.items():
        print(f"\n{comp}:")
        if data.get("max_temperature") is not None:
            print(f"  Maximum Temperature: {data['max_temperature']:.2f} °C")
        if data.get("avg_temperature") is not None:
            print(f"  Average Temperature: {data['avg_temperature']:.2f} °C")
    
    # --- Cleanup ---
    ipk.save_project()
    desktop.release_desktop()

if __name__ == "__main__":
    run_analysis()


In [ ]:
import os
import tempfile
from pyaedt import Icepak, Desktop

class Package2_5D_Analysis:
    def __init__(self, version="2024.1"):
        """Initialize AEDT desktop and Icepak design."""
        self.project_name = "Package2_5D"
        self.design_name = "ThermalAnalysis"
        # Launch AEDT Desktop in non-graphical mode.
        self.desktop = Desktop(version, non_graphical=True, new_desktop=True)
        # Create a new Icepak project.
        self.ipk = Icepak(project=self.project_name, design=self.design_name,
                          solution_type="SteadyState", version=version)
        # Set model units.
        self.ipk.modeler.model_units = "mm"
        # Define chiplet configurations.
        self.chiplet_configs = [
            {"pos": [5, 5, 1.95], "size": [5, 5, 0.5], "power": 50},
            {"pos": [15, 5, 1.95], "size": [8, 4, 0.5], "power": 75},
            {"pos": [10, 15, 1.95], "size": [6, 6, 0.5], "power": 60}
        ]
        self.chiplets = []  # Will store chiplet object info

    def setup_materials(self):
        """Define custom materials."""
        self.ipk.materials.add_material(
            name="silicon_custom",
            properties={
                "Thermal Conductivity": "130",  # W/m-K
                "Mass Density": "2329",         # kg/m³
                "Specific Heat": "700"          # J/kg-K
            }
        )
        self.ipk.materials.add_material(
            name="solder_custom",
            properties={
                "Thermal Conductivity": "50",   # W/m-K
                "Mass Density": "7370",         # kg/m³
                "Specific Heat": "150"          # J/kg-K
            }
        )
        self.ipk.materials.add_material(
            name="tim_custom",
            properties={
                "Thermal Conductivity": "5",    # W/m-K
                "Mass Density": "3000",         # kg/m³
                "Specific Heat": "1000"         # J/kg-K
            }
        )

    def create_geometry(self):
        """Create the package geometry."""
        substrate = self.ipk.modeler.create_box(
            origin=[0, 0, 0],
            sizes=[30, 30, 1],
            name="substrate",
            material="fr4_epoxy"
        )
        interposer = self.ipk.modeler.create_box(
            origin=[2.5, 2.5, 1.1],
            sizes=[25, 25, 0.75],
            name="interposer",
            material="silicon_custom"
        )
        # Create chiplets.
        for i, config in enumerate(self.chiplet_configs):
            chiplet_name = f"chiplet_{i}"
            chiplet = self.ipk.modeler.create_box(
                origin=config["pos"],
                sizes=config["size"],
                name=chiplet_name,
                material="silicon_custom"
            )
            self.chiplets.append({"name": chiplet_name, "power": config["power"]})
        # Create TIM1.
        tim1 = self.ipk.modeler.create_box(
            origin=[0, 0, 2.45],
            sizes=[30, 30, 0.05],
            name="tim1",
            material="tim_custom"
        )
        # Create heat spreader.
        heatspreader = self.ipk.modeler.create_box(
            origin=[-2.5, -2.5, 2.5],
            sizes=[35, 35, 1],
            name="heatsink1",  # Note: later used as target for boundary conditions and temperature data.
            material="copper"
        )
        # Create TIM2.
        tim2 = self.ipk.modeler.create_box(
            origin=[-2.5, -2.5, 3.5],
            sizes=[35, 35, 0.05],
            name="tim2",
            material="tim_custom"
        )
        # Create heatsink using parametric fin heat sink.
        # (The parameter names follow the new API; adjust values as needed.)
        heatsink = self.ipk.create_parametric_fin_heat_sink(
            hs_height=2,                # base height (thickness)
            hs_width=40,                # base width
            hs_basethick=1,             # base thickness
            pitch=2,                    # pitch between fins
            thick=1,                    # fin thickness
            length=40,                  # fin length
            height=10,                  # fin height
            draftangle=0,
            patternangle=10,
            separation=2,               # fin separation
            symmetric=True,
            symmetric_separation=20,
            numcolumn_perside=2,
            vertical_separation=10,
            material="aluminum",
            center=[-5, -5, 3.55],
            plane_enum=self.ipk.PLANE.XY,
            rotation=0,
            tolerance=0.001
        )

    def setup_thermal_loads(self):
        """Setup thermal loads and monitors on chiplets."""
        for idx, chiplet in enumerate(self.chiplets):
            # Assign a thermal source to the chiplet.
            self.ipk.assign_source(chiplet["name"], "Total Power", chiplet["power"])
            # Compute the chiplet's center point.
            config = self.chiplet_configs[idx]
            pos = config["pos"]
            size = config["size"]
            center = [pos[0] + size[0] / 2,
                      pos[1] + size[1] / 2,
                      pos[2] + size[2] / 2]
            monitor_name = f"temp_monitor_{chiplet['name']}"
            # Instead of using the old create_temperature_monitor method,
            # assign a point monitor via the post processor.
            #self.ipk.post.assign_point_monitor(chiplet["name"], monitor_name, center)
            region = self.ipk.modeler["Region"]
            self.ipk.assign_pressure_free_opening(
                assignment=region.top_face_x.id, boundary_name="Outlet"
            )
            self.ipk.assign_velocity_free_opening(
                assignment=region.bottom_face_x.id,
                boundary_name="Inlet",
                velocity=["1m_per_sec", "0m_per_sec", "0m_per_sec"],
            )

    def setup_boundary_conditions(self):
        """Setup boundary conditions for the design."""
        # Set ambient temperature.
        
        self.ipk.set_temperature(25)
        # Apply convection to the heatsink.
        self.ipk.assign_boundary_condition("heatsink1", "Convection", {"HTC": 20, "Temperature": 25})
        # Apply radiation to the heatsink.
        self.ipk.assign_boundary_condition("heatsink1", "Radiation", {"Emissivity": 0.9})

    def setup_mesh(self):
        """Define mesh settings."""
        self.ipk.mesh.set_global_mesh_settings(max_element_size=2, min_element_size=0.1)
        for chiplet in self.chiplets:
            self.ipk.mesh.assign_mesh_control(chiplet["name"], max_element_size=0.2)
        self.ipk.mesh.assign_mesh_control("heatsink1", max_element_size=0.5)

    def run_analysis(self):
        """Create setup (if needed) and run analysis."""
        if not self.ipk.setup_names:
            setup = self.ipk.create_setup()
            setup.props["Convergence Criteria"] = 1e-6
            setup.props["Max Number Of Iterations"] = 1000
        self.ipk.analyze_all()

    def get_results(self):
        """Extract analysis results."""
        results = {}
        # Get temperature data from chiplets.
        for chiplet in self.chiplets:
            name = chiplet["name"]
            monitor_name = f"temp_monitor_{name}"
            temp_data = self.ipk.post.get_temperature_monitor_data(monitor_name)
            results[name] = {
                "max_temperature": max(temp_data) if temp_data else None,
                "power_density": chiplet["power"]
            }
        # Get temperature data from the heatsink.
        heatsink_temp = self.ipk.post.get_temperature_data("heatsink1")
        if heatsink_temp:
            results["heatsink"] = {
                "max_temperature": max(heatsink_temp),
                "min_temperature": min(heatsink_temp),
                "avg_temperature": sum(heatsink_temp) / len(heatsink_temp)
            }
        return results

    def cleanup(self):
        """Save project and release the desktop."""
        self.ipk.save_project()
        self.desktop.release_desktop()

    def plot_visualization(self, temp_folder):
        """Plot visualization of results."""
        
        plot1 = self.ipk.plot(
            show=False,
            output_file=os.path.join(temp_folder.name, "Graphics_card_1.jpg"),
            plot_air_objects=False,
        )

        self.ipk.modeler.rotate(ipk.modeler.object_names, "X")

        plot2 = self.ipk.plot(
            show=False,
            output_file=os.path.join(temp_folder.name, "Graphics_card_2.jpg"),
            plot_air_objects=False,
        )



def run_analysis():
    analysis = None
    try:
        analysis = Package2_5D_Analysis()
        analysis.setup_materials()
        analysis.create_geometry()
        temp_folder = tempfile.TemporaryDirectory(suffix=".ansys")
        analysis.plot_visualization(temp_folder)

        analysis.setup_thermal_loads()
        analysis.setup_boundary_conditions()
        analysis.setup_mesh()
        analysis.run_analysis()
        results = analysis.get_results()
        print("\nThermal Analysis Results:")
        for component, data in results.items():
            print(f"\n{component}:")
            if "max_temperature" in data and data["max_temperature"] is not None:
                print(f"  Maximum Temperature: {data['max_temperature']:.2f}°C")
            if "power_density" in data:
                print(f"  Power Density: {data['power_density']} W/m²")
            if "avg_temperature" in data:
                print(f"  Average Temperature: {data['avg_temperature']:.2f}°C")
    except Exception as e:
        print(f"Error during analysis: {str(e)}")
    finally:
        if analysis:
            analysis.cleanup()

if __name__ == "__main__":
    run_analysis()

In [ ]:
app =Icepak
app.modeler.create_air_region()

In [ ]:
setup=app.create_setup(setup_type="IcepakSteadyState")

In [ ]:
import os
import tempfile
from pyaedt import Icepak, Desktop

class Package2_5D_Analysis:
    def __init__(self, version="2024.1"):
        """Initialize AEDT desktop and Icepak design."""
        self.project_name = "Package2_5D"
        self.design_name = "ThermalAnalysis"
        # Launch AEDT Desktop in non-graphical mode.
        self.desktop = Desktop(version, non_graphical=True, new_desktop=True)
        # Create a new Icepak project.
        self.ipk = Icepak(project=self.project_name, design=self.design_name,
                          solution_type="SteadyState", version=version)
        # Set model units.
        self.ipk.modeler.model_units = "mm"
        # Define chiplet configurations.
        self.chiplet_configs = [
            {"pos": [5, 5, 1.95], "size": [5, 5, 0.5], "power": 50},
            {"pos": [15, 5, 1.95], "size": [8, 4, 0.5], "power": 75},
            {"pos": [10, 15, 1.95], "size": [6, 6, 0.5], "power": 60}
        ]
        self.chiplets = []  # Will store chiplet object info

    def setup_materials(self):
        """Define custom materials."""
        self.ipk.materials.add_material(
            name="silicon_custom",
            properties={
                "Thermal Conductivity": "130",  # W/m-K
                "Mass Density": "2329",         # kg/m³
                "Specific Heat": "700"          # J/kg-K
            }
        )
        self.ipk.materials.add_material(
            name="solder_custom",
            properties={
                "Thermal Conductivity": "50",   # W/m-K
                "Mass Density": "7370",         # kg/m³
                "Specific Heat": "150"          # J/kg-K
            }
        )
        self.ipk.materials.add_material(
            name="tim_custom",
            properties={
                "Thermal Conductivity": "5",    # W/m-K
                "Mass Density": "3000",         # kg/m³
                "Specific Heat": "1000"         # J/kg-K
            }
        )

    def create_geometry(self):
        """Create the package geometry."""
        substrate = self.ipk.modeler.create_box(
            origin=[0, 0, 0],
            sizes=[30, 30, 1],
            name="substrate",
            material="fr4_epoxy"
        )
        interposer = self.ipk.modeler.create_box(
            origin=[2.5, 2.5, 1.1],
            sizes=[25, 25, 0.75],
            name="interposer",
            material="silicon_custom"
        )
        # Create chiplets.
        for i, config in enumerate(self.chiplet_configs):
            chiplet_name = f"chiplet_{i}"
            chiplet = self.ipk.modeler.create_box(
                origin=config["pos"],
                sizes=config["size"],
                name=chiplet_name,
                material="silicon_custom"
            )
            self.chiplets.append({"name": chiplet_name, "power": config["power"]})
        # Create TIM1.
        tim1 = self.ipk.modeler.create_box(
            origin=[0, 0, 2.45],
            sizes=[30, 30, 0.05],
            name="tim1",
            material="tim_custom"
        )
        # Create heat spreader.
        heatspreader = self.ipk.modeler.create_box(
            origin=[-2.5, -2.5, 2.5],
            sizes=[35, 35, 1],
            name="heatsink1",  # Note: later used as target for boundary conditions and temperature data.
            material="copper"
        )
        # Create TIM2.
        tim2 = self.ipk.modeler.create_box(
            origin=[-2.5, -2.5, 3.5],
            sizes=[35, 35, 0.05],
            name="tim2",
            material="tim_custom"
        )
        # Create heatsink using parametric fin heat sink.
        # (The parameter names follow the new API; adjust values as needed.)
        heatsink = self.ipk.create_parametric_fin_heat_sink(
            hs_height=2,                # base height (thickness)
            hs_width=40,                # base width
            hs_basethick=1,             # base thickness
            pitch=2,                    # pitch between fins
            thick=1,                    # fin thickness
            length=40,                  # fin length
            height=10,                  # fin height
            draftangle=0,
            patternangle=10,
            separation=2,               # fin separation
            symmetric=True,
            symmetric_separation=20,
            numcolumn_perside=2,
            vertical_separation=10,
            material="aluminum",
            center=[-5, -5, 3.55],
            plane_enum=self.ipk.PLANE.XY,
            rotation=0,
            tolerance=0.001
        )

    def setup_thermal_loads(self):
        """Setup thermal loads and monitors on chiplets."""
        for idx, chiplet in enumerate(self.chiplets):
            # Assign a thermal source to the chiplet.
            self.ipk.assign_source(chiplet["name"], "Total Power", chiplet["power"])
            # Compute the chiplet's center point.
            config = self.chiplet_configs[idx]
            pos = config["pos"]
            size = config["size"]
            center = [pos[0] + size[0] / 2,
                      pos[1] + size[1] / 2,
                      pos[2] + size[2] / 2]
            monitor_name = f"temp_monitor_{chiplet['name']}"
            # Instead of using the old create_temperature_monitor method,
            # assign a point monitor via the post processor.
            self.ipk.post.assign_point_monitor(chiplet["name"], monitor_name, center)

    '''
    def setup_boundary_conditions(self):
        """Setup boundary conditions for the design."""
        # Set ambient temperature.
        self.ipk.set_temperature(25)
        # Apply convection to the heatsink.
        self.ipk.assign_boundary_condition("heatsink1", "Convection", {"HTC": 20, "Temperature": 25})
        # Apply radiation to the heatsink.
        self.ipk.assign_boundary_condition("heatsink1", "Radiation", {"Emissivity": 0.9})
    '''
    def setup_mesh(self):
        """Define mesh settings."""
        self.ipk.mesh.set_global_mesh_settings(max_element_size=2, min_element_size=0.1)
        for chiplet in self.chiplets:
            self.ipk.mesh.assign_mesh_control(chiplet["name"], max_element_size=0.2)
        self.ipk.mesh.assign_mesh_control("heatsink1", max_element_size=0.5)

    def run_analysis(self):
        """Create setup (if needed) and run analysis."""
        if not self.ipk.setup_names:
            setup = self.ipk.create_setup()
            setup.props["Convergence Criteria"] = 1e-6
            setup.props["Max Number Of Iterations"] = 1000
        self.ipk.analyze_all()

    def get_results(self):
        """Extract analysis results."""
        results = {}
        # Get temperature data from chiplets.
        for chiplet in self.chiplets:
            name = chiplet["name"]
            monitor_name = f"temp_monitor_{name}"
            temp_data = self.ipk.post.get_temperature_monitor_data(monitor_name)
            results[name] = {
                "max_temperature": max(temp_data) if temp_data else None,
                "power_density": chiplet["power"]
            }
        # Get temperature data from the heatsink.
        heatsink_temp = self.ipk.post.get_temperature_data("heatsink1")
        if heatsink_temp:
            results["heatsink"] = {
                "max_temperature": max(heatsink_temp),
                "min_temperature": min(heatsink_temp),
                "avg_temperature": sum(heatsink_temp) / len(heatsink_temp)
            }
        return results

    def cleanup(self):
        """Save project and release the desktop."""
        self.ipk.save_project()
        self.desktop.release_desktop()

def run_analysis():
    analysis = None
    try:
        analysis = Package2_5D_Analysis()
        analysis.setup_materials()
        analysis.create_geometry()
        analysis.setup_thermal_loads()
        #analysis.setup_boundary_conditions()
        analysis.setup_mesh()
        analysis.run_analysis()
        results = analysis.get_results()
        print("\nThermal Analysis Results:")
        for component, data in results.items():
            print(f"\n{component}:")
            if "max_temperature" in data and data["max_temperature"] is not None:
                print(f"  Maximum Temperature: {data['max_temperature']:.2f}°C")
            if "power_density" in data:
                print(f"  Power Density: {data['power_density']} W/m²")
            if "avg_temperature" in data:
                print(f"  Average Temperature: {data['avg_temperature']:.2f}°C")
    except Exception as e:
        print(f"Error during analysis: {str(e)}")
    finally:
        if analysis:
            analysis.cleanup()

if __name__ == "__main__":
    run_analysis()


In [ ]:
app.get_object_material_properties(1)

In [ ]:
analysis.cleanup()

In [ ]:
from pyaedt import Mechanical
from pyaedt.generic.constants import PLANE

def mm_to_meter(val_mm):
    """Convert millimeters to meters."""
    return val_mm * 0.001

# Define stack layers (bottom to top)
layers = [
    {"name": "Substrate",    "length": 35.0, "width": 35.0, "thickness": 0.5, "material": "fr4_epoxy"},
    {"name": "C4_Bumps",     "length": 30.0, "width": 30.0, "thickness": 0.1, "material": "solder"},
    {"name": "Interposer",   "length": 25.0, "width": 25.0, "thickness": 0.3, "material": "silicon"},
    {"name": "MicroBumps",   "length": 25.0, "width": 25.0, "thickness": 0.05, "material": "solder"},
    {"name": "TIM_pads",     "length": 30.0, "width": 30.0, "thickness": 0.05, "material": "thermal_interface"},
    {"name": "HeatSpreader", "length": 35.0, "width": 35.0, "thickness": 0.5, "material": "copper"},
    {"name": "TIM2_pads",    "length": 35.0, "width": 35.0, "thickness": 0.05, "material": "thermal_interface"},
    {"name": "HeatSink",     "length": 40.0, "width": 40.0, "thickness": 2.0, "material": "aluminum"}
]

# Define chiplets (placed after "MicroBumps")
chiplets = [
    {"name": "Chiplet_1", "x_offset": 5.0, "y_offset": 5.0, "length": 5.0, "width": 5.0, "thickness": 0.2, "material": "silicon", "heat_generation": 50},
    {"name": "Chiplet_2", "x_offset": 10.0, "y_offset": 5.0, "length": 4.0, "width": 4.0, "thickness": 0.2, "material": "silicon", "heat_generation": 40},
    {"name": "Chiplet_3", "x_offset": 15.0, "y_offset": 5.0, "length": 6.0, "width": 6.0, "thickness": 0.2, "material": "silicon", "heat_generation": 60}
]

def main():
    # Initialize AEDT Mechanical
    mech = Mechanical(specified_version="2024.1")
    
    # Get the active design
    design = mech.modeler
    
    # Find maximum dimensions for centering
    max_length = max(layer["length"] for layer in layers)
    max_width = max(layer["width"] for layer in layers)
    
    # Track current z position
    current_z_mm = 0.0
    
    # Create each layer
    for layer in layers:
        # Calculate position
        x1 = (max_length - layer["length"]) / 2.0
        y1 = (max_width - layer["width"]) / 2.0
        z1 = current_z_mm
        
        if layer["name"] == "HeatSink":
            # Create heatsink with fins
            base_thickness_mm = 1.0
            fin_height_mm = 6.0
            
            # Create base
            base_name = f"{layer['name']}_Base"
            base = design.create_box(
                position=[mm_to_meter(x1), mm_to_meter(y1), mm_to_meter(z1)],
                dimensions=[
                    mm_to_meter(layer["length"]),
                    mm_to_meter(layer["width"]),
                    mm_to_meter(base_thickness_mm)
                ],
                name=base_name
            )
            
            # Add fins
            fin_count = 13
            fin_thickness = 1.0
            fin_gap = 2.0
            margin_x = 2.0
            
            fin_start_x = x1 + margin_x
            current_x = fin_start_x
            
            for fin_idx in range(fin_count):
                if current_x + fin_thickness > (x1 + layer["length"] - margin_x):
                    break
                    
                fin_name = f"{layer['name']}_Fin_{fin_idx + 1}"
                fin = design.create_box(
                    position=[
                        mm_to_meter(current_x),
                        mm_to_meter(y1),
                        mm_to_meter(z1 + base_thickness_mm)
                    ],
                    dimensions=[
                        mm_to_meter(fin_thickness),
                        mm_to_meter(layer["width"]),
                        mm_to_meter(fin_height_mm)
                    ],
                    name=fin_name
                )
                
                # Set material
                design.assign_material(fin_name, layer["material"])
                current_x += (fin_thickness + fin_gap)
                
            # Set material for base
            design.assign_material(base_name, layer["material"])
            current_z_mm += layer["thickness"]
            
        else:
            # Create regular layer
            body_name = f"{layer['name']}_Body"
            body = design.create_box(
                position=[mm_to_meter(x1), mm_to_meter(y1), mm_to_meter(z1)],
                dimensions=[
                    mm_to_meter(layer["length"]),
                    mm_to_meter(layer["width"]),
                    mm_to_meter(layer["thickness"])
                ],
                name=body_name
            )
            
            # Set material
            design.assign_material(body_name, layer["material"])
            current_z_mm += layer["thickness"]
        
        # Add chiplets after MicroBumps layer
        if layer["name"] == "MicroBumps":
            max_chip_thickness_mm = 0.0
            
            for chip in chiplets:
                chip_x1 = x1 + chip["x_offset"]
                chip_y1 = y1 + chip["y_offset"]
                chip_z1 = current_z_mm
                
                chip_name = f"{chip['name']}_Body"
                chiplet = design.create_box(
                    position=[mm_to_meter(chip_x1), mm_to_meter(chip_y1), mm_to_meter(chip_z1)],
                    dimensions=[
                        mm_to_meter(chip["length"]),
                        mm_to_meter(chip["width"]),
                        mm_to_meter(chip["thickness"])
                    ],
                    name=chip_name
                )
                
                # Set material and heat generation for chiplet
                design.assign_material(chip_name, chip["material"])
                
                # Add heat generation to chiplet
                mech.assign_power_density(chip_name, chip["heat_generation"])
                
                max_chip_thickness_mm = max(max_chip_thickness_mm, chip["thickness"])
            
            current_z_mm += max_chip_thickness_mm
    
    # Set up thermal-mechanical analysis
    mech.change_solution_type("Steady-State Thermal")
    
    # Add thermal boundary conditions
    # Ambient temperature at top of heatsink
    mech.assign_temperature(f"{layers[-1]['name']}_Base", 25)  # 25°C ambient
    
    # Add convection to heatsink fins
    for i in range(fin_count):
        fin_name = f"HeatSink_Fin_{i + 1}"
        mech.assign_convection(fin_name, 25, 50)  # 25°C ambient, 50 W/m²·K coefficient
    
    # Save the project
    mech.save_project("thermal_mechanical_analysis.aedt")

if __name__ == "__main__":
    main()

In [ ]:
from pyaedt import Mechanical
from pyaedt.generic.constants import PLANE

def mm_to_meter(val_mm):
    """Convert millimeters to meters."""
    return val_mm * 0.001

# Define stack layers (bottom to top)
layers = [
    {"name": "Substrate",    "length": 35.0, "width": 35.0, "thickness": 0.5, "material": "FR4_Epoxy"},
    {"name": "C4_Bumps",     "length": 30.0, "width": 30.0, "thickness": 0.1, "material": "Solder"},
    {"name": "Interposer",   "length": 25.0, "width": 25.0, "thickness": 0.3, "material": "Silicon"},
    {"name": "MicroBumps",   "length": 25.0, "width": 25.0, "thickness": 0.05, "material": "Solder"},
    {"name": "TIM_pads",     "length": 30.0, "width": 30.0, "thickness": 0.05, "material": "Thermal_Interface"},
    {"name": "HeatSpreader", "length": 35.0, "width": 35.0, "thickness": 0.5, "material": "Copper"},
    {"name": "TIM2_pads",    "length": 35.0, "width": 35.0, "thickness": 0.05, "material": "Thermal_Interface"},
    {"name": "HeatSink",     "length": 40.0, "width": 40.0, "thickness": 2.0, "material": "Aluminum"}
]

# Define chiplets (placed after "MicroBumps")
chiplets = [
    {"name": "Chiplet_1", "x_offset": 5.0, "y_offset": 5.0, "length": 5.0, "width": 5.0, "thickness": 0.2, "material": "Silicon", "heat_generation": 50},
    {"name": "Chiplet_2", "x_offset": 10.0, "y_offset": 5.0, "length": 4.0, "width": 4.0, "thickness": 0.2, "material": "Silicon", "heat_generation": 40},
    {"name": "Chiplet_3", "x_offset": 15.0, "y_offset": 5.0, "length": 6.0, "width": 6.0, "thickness": 0.2, "material": "Silicon", "heat_generation": 60}
]

def main():
    # Initialize AEDT Mechanical
    mech = Mechanical(specified_version="2024.1")
    
    # Get the active design and modeler
    design = mech.modeler
    
    # Find maximum dimensions for centering
    max_length = max(layer["length"] for layer in layers)
    max_width = max(layer["width"] for layer in layers)
    
    # Track current z position
    current_z_mm = 0.0
    
    # Create each layer
    for layer in layers:
        # Calculate position
        x1 = (max_length - layer["length"]) / 2.0
        y1 = (max_width - layer["width"]) / 2.0
        z1 = current_z_mm
        
        if layer["name"] == "HeatSink":
            # Create heatsink with fins
            base_thickness_mm = 1.0
            fin_height_mm = 6.0
            
            # Create base
            base_name = f"{layer['name']}_Base"
            base = design.create_box(
                position=[mm_to_meter(x1), mm_to_meter(y1), mm_to_meter(z1)],
                sizes=[
                    mm_to_meter(layer["length"]),
                    mm_to_meter(layer["width"]),
                    mm_to_meter(base_thickness_mm)
                ],
                name=base_name
            )
            
            # Add fins
            fin_count = 13
            fin_thickness = 1.0
            fin_gap = 2.0
            margin_x = 2.0
            
            fin_start_x = x1 + margin_x
            current_x = fin_start_x
            
            for fin_idx in range(fin_count):
                if current_x + fin_thickness > (x1 + layer["length"] - margin_x):
                    break
                    
                fin_name = f"{layer['name']}_Fin_{fin_idx + 1}"
                fin = design.create_box(
                    position=[
                        mm_to_meter(current_x),
                        mm_to_meter(y1),
                        mm_to_meter(z1 + base_thickness_mm)
                    ],
                    sizes=[
                        mm_to_meter(fin_thickness),
                        mm_to_meter(layer["width"]),
                        mm_to_meter(fin_height_mm)
                    ],
                    name=fin_name,
                    material=layer["material"]
                )
                
                # Set material using the mechanical object
                
                current_x += (fin_thickness + fin_gap)
                
            # Set material for base
            #mech.materials.assign_to_bodies(base_name, layer["material"])
            current_z_mm += layer["thickness"]
            
        else:
            # Create regular layer
            body_name = f"{layer['name']}_Body"
            body = design.create_box(
                position=[mm_to_meter(x1), mm_to_meter(y1), mm_to_meter(z1)],
                sizes=[
                    mm_to_meter(layer["length"]),
                    mm_to_meter(layer["width"]),
                    mm_to_meter(layer["thickness"])
                ],
                name=body_name,
                material=layer["material"]
            )
            
            # Set material using the mechanical object
            current_z_mm += layer["thickness"]
        
        # Add chiplets after MicroBumps layer
        if layer["name"] == "MicroBumps":
            max_chip_thickness_mm = 0.0
            
            for chip in chiplets:
                chip_x1 = x1 + chip["x_offset"]
                chip_y1 = y1 + chip["y_offset"]
                chip_z1 = current_z_mm
                
                chip_name = f"{chip['name']}_Body"
                chiplet = design.create_box(
                    position=[mm_to_meter(chip_x1), mm_to_meter(chip_y1), mm_to_meter(chip_z1)],
                    sizes=[
                        mm_to_meter(chip["length"]),
                        mm_to_meter(chip["width"]),
                        mm_to_meter(chip["thickness"])
                    ],
                    name=chip_name,
                    material=chip["material"]
                )
                
                # Add heat generation to chiplet
                mech.assign_heat_generation(chip_name, chip["heat_generation"])
                
                max_chip_thickness_mm = max(max_chip_thickness_mm, chip["thickness"])
            
            current_z_mm += max_chip_thickness_mm
    
    # Set up thermal-mechanical analysis
    mech.solution_type = "Steady-State Thermal"
    
    # Add thermal boundary conditions
    # Ambient temperature at top of heatsink
    #mech.assign_uniform_temperature(f"{layers[-1]['name']}_Base", 25)  # 25°C ambient

    
    # Add convection to heatsink fins
    for i in range(fin_count):
        fin_name = f"HeatSink_Fin_{i + 1}"
        mech.assign_uniform_convection(assignment=fin_name, convection_unit="w_per_m2kel",convection_value=200,temperature="AmbientTemp")  # 25°C ambient, 50 W/m²·K coefficient
    
    # Save the project
    mech.save_project("thermal_mechanical_analysis.aedt")

if __name__ == "__main__":
    main()